### Подготовка окружения

```bash
pip -q install pandas numpy scikit-learn
```


In [10]:
import numpy as np
import pandas as pd

from dataclasses import dataclass

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

RANDOM_STATE = 42
TEST_SIZE = 0.2

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)
pd.set_option("display.precision", 4)


### Вспомогательные функции

Ниже — общая функция, которая:
- делит признаки на числовые/категориальные
- строит общий препроцессинг (импутация + one-hot + scaling)
- обучает 4 модели (LogReg, SVM, Tree, kNN)
- считает метрики на тестовой выборке


In [11]:
@dataclass
class ModelResult:
    model: str
    accuracy: float
    precision: float
    recall: float
    f1: float


def _infer_feature_types(X: pd.DataFrame):
    numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]
    return numeric_cols, categorical_cols


def evaluate_models(
    df: pd.DataFrame,
    target_col: str,
    positive_label=None,
    stratify: bool = True,
):
    X = df.drop(columns=[target_col])
    y = df[target_col]

    numeric_cols, categorical_cols = _infer_feature_types(X)

    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_cols),
            ("cat", categorical_transformer, categorical_cols),
        ],
        remainder="drop",
    )

    models = {
        "LogisticRegression": LogisticRegression(max_iter=2000, class_weight=None, random_state=RANDOM_STATE),
        "SVM (RBF)": SVC(kernel="rbf", C=1.0, gamma="scale"),
        "DecisionTree": DecisionTreeClassifier(random_state=RANDOM_STATE),
        "kNN": KNeighborsClassifier(n_neighbors=5),
    }

    strat = y if stratify else None
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=strat,
    )

    results = []
    for name, clf in models.items():
        pipe = Pipeline(steps=[("preprocess", preprocessor), ("model", clf)])
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)

        if positive_label is None:
            # multiclass или бинарная без явно заданного "positive"
            precision = precision_score(y_test, y_pred, average="macro", zero_division=0)
            recall = recall_score(y_test, y_pred, average="macro", zero_division=0)
            f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
        else:
            precision = precision_score(y_test, y_pred, pos_label=positive_label, zero_division=0)
            recall = recall_score(y_test, y_pred, pos_label=positive_label, zero_division=0)
            f1 = f1_score(y_test, y_pred, pos_label=positive_label, zero_division=0)

        acc = accuracy_score(y_test, y_pred)
        results.append(ModelResult(name, acc, precision, recall, f1))

    res_df = pd.DataFrame([r.__dict__ for r in results]).sort_values(by=["f1", "accuracy"], ascending=False)
    return res_df


## Датасет 1: Голосования конгрессменов

### Загрузка и предобработка
- Признаки: 16 голосований (категориальные: `y/n/?`)
- Целевая переменная: `political_party` (2 класса)
- Символ `?` трактуем как пропуск


In [12]:
voting_path = "congressional_voting_dataset.csv"
df_vote = pd.read_csv(voting_path)

# '?' → NaN
df_vote = df_vote.replace("?", np.nan)

display(df_vote.head())
print("Shape:", df_vote.shape)
print("Target distribution:\n", df_vote["political_party"].value_counts())


,handicapped_infants,water_project_cost_sharing,adoption_of_the_budget_resolution,physician_fee_freeze,el_salvador_aid,religious_groups_in_schools,anti_satellite_test_ban,aid_to_nicaraguan_contras,mx_missile,immigration,synfuels_corporation_cutback,education_spending,superfund_right_to_sue,crime,duty_free_exports,export_administration_act_south_africa,political_party
0,n,y,n,y,y,y,n,n,n,y,NaN,y,y,y,n,y,republican
1,n,y,n,y,y,y,n,n,n,n,n,y,y,y,n,NaN,republican
2,NaN,y,y,NaN,y,y,n,n,n,n,y,n,y,y,n,n,democrat
3,n,y,y,n,NaN,y,n,n,n,n,y,n,y,n,n,y,democrat
4,y,y,y,n,y,y,n,n,n,n,y,NaN,y,y,y,y,democrat


Shape: (435, 17)
Target distribution:
 political_party
democrat      267
republican    168
Name: count, dtype: int64


### Обучение и сравнение моделей (метрики на тесте)

Для бинарной классификации считаем precision/recall/f1 по позитивному классу. Здесь позитивный класс условно зададим как `republican`.

In [13]:
vote_metrics = evaluate_models(df_vote, target_col="political_party", positive_label="republican", stratify=True)
display(vote_metrics)

best_vote = vote_metrics.iloc[0]
print("Best (by F1 then accuracy):", best_vote["model"], "| f1=", round(best_vote["f1"], 4), "acc=", round(best_vote["accuracy"], 4))


,model,accuracy,precision,recall,f1
1,SVM (RBF),0.9655,0.9429,0.9706,0.9565
2,DecisionTree,0.9655,0.9697,0.9412,0.9552
0,LogisticRegression,0.9540,0.9412,0.9412,0.9412
3,kNN,0.9540,0.9412,0.9412,0.9412


Best (by F1 then accuracy): SVM (RBF) | f1= 0.9565 acc= 0.9655


### Выводы по датасету голосований (текст)

- **Качество**: сравните значения accuracy/precision/recall/f1 в таблице выше.
- **Лучший алгоритм**: выбран автоматически как максимум по **F1**, при равенстве — по accuracy.
- **Почему могли отличаться метрики**: данные категориальные, много пропусков (`?`), классы могут быть несбалансированы; некоторые модели сильнее выигрывают от масштабирования/one-hot.


## Датасет 2: Болезнь сердца (Cleveland)

### Загрузка и предобработка
Целевая переменная `num`: 0 — нет болезни; 1..4 — болезнь. Преобразуем к бинарной: `target = 1`, если `num > 0`.

Также в наборе встречаются аномальные значения `-100000` (в этом файле) — будем считать их пропусками.

In [14]:
heart_path = "heart_disease_dataset.csv"
df_heart = pd.read_csv(heart_path)

# -100000 → NaN (как "код" пропусков)
df_heart = df_heart.replace(-100000, np.nan)

# Бинаризация таргета
df_heart["target"] = (df_heart["num"] > 0).astype(int)
df_heart = df_heart.drop(columns=["num"])

display(df_heart.head())
print("Shape:", df_heart.shape)
print("Target distribution:\n", df_heart["target"].value_counts())


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0,1
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0,1
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0,0


Shape: (303, 14)
Target distribution:
 target
0    164
1    139
Name: count, dtype: int64


### Обучение и сравнение моделей (метрики на тесте)

Позитивный класс: `1` (болезнь присутствует).

In [15]:
heart_metrics = evaluate_models(df_heart, target_col="target", positive_label=1, stratify=True)
display(heart_metrics)

best_heart = heart_metrics.iloc[0]
print("Best (by F1 then accuracy):", best_heart["model"], "| f1=", round(best_heart["f1"], 4), "acc=", round(best_heart["accuracy"], 4))


,model,accuracy,precision,recall,f1
3,kNN,0.8852,0.8000,1.0000,0.8889
0,LogisticRegression,0.8689,0.8125,0.9286,0.8667
1,SVM (RBF),0.8525,0.8065,0.8929,0.8475
2,DecisionTree,0.7213,0.6571,0.8214,0.7302


Best (by F1 then accuracy): kNN | f1= 0.8889 acc= 0.8852


### Выводы по датасету болезни сердца (текст)

- **Предобработка**: пропуски заполнены (числовые — медианой, категориальные — модой), категориальные признаки закодированы one-hot, числовые стандартизированы.
- **Лучший алгоритм**: выбран по максимуму **F1** (т.к. важен баланс precision/recall), при равенстве — по accuracy.
- **Практический смысл**: при медицинских задачах часто важнее не только accuracy, а именно recall по классу "болезнь" (не пропускать больных) и разумный precision (не перегружать ложными срабатываниями).


## Итоговый выбор

Здесь можно явно (словами) зафиксировать итог:
- какой алгоритм лучший на каждом датасете
- какие метрики были ключевыми (например, F1 или recall)
- кратко объяснить причины (тип признаков, размер данных, нелинейность)
